# Notebook 01: Database Preparation

This notebook locates the raw NetFlow dataset, creates reusable Parquet outputs, creates an optional DuckDB database, and produces a database preparation summary.

## Short Problem Statement

Existing deep learning-based **Intrusion Detection Systems (IDS)** often rely mainly on time-domain flow features and may miss hidden burst patterns, repeated traffic rhythms, and frequency-based attack signatures. **SentinelFlow** addresses this by using **Fast Fourier Transform (FFT)**-enhanced traffic profiling to improve deep learning-based intrusion detection on large-scale network traffic datasets.

In [ ]:
from pathlib import Path
import sys, json, os, time
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'sentinelflow_utils.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from sentinelflow_utils import *
ensure_dirs(PROJECT_ROOT)
print('PROJECT_ROOT:', PROJECT_ROOT)

In [ ]:
# Configuration
# Leave RAW_DATA_PATH blank to automatically use the first usable file in data/raw.
RAW_DATA_PATH = ''
MAX_ROWS_FOR_DEMO = None  # Set to an integer such as 1_000_000 for quick runs.

raw_path = find_raw_dataset_path(PROJECT_ROOT, RAW_DATA_PATH)
print('Raw dataset:', raw_path)

In [ ]:
started = stage_message('Reading raw table')
df_raw = read_table_safely(raw_path, max_rows=MAX_ROWS_FOR_DEMO)
stage_message(f'Loaded raw table with shape {df_raw.shape}', started)
df_raw.head()

In [ ]:
started = stage_message('Normalizing and saving raw string-safe Parquet')
df_raw = normalize_column_names(df_raw)
raw_parquet_path = PROJECT_ROOT / 'data/processed/sentinelflow_raw_netflow_string.parquet'
df_raw.to_parquet(raw_parquet_path, index=False)
stage_message(f'Saved {raw_parquet_path}', started)
print('Columns:', len(df_raw.columns))
print(df_raw.columns.tolist()[:20])

In [ ]:
started = stage_message('Cleaning dataset for analysis')
df_clean, clean_summary = clean_netflow_df(df_raw)
analysis_path = PROJECT_ROOT / 'data/processed/sentinelflow_analysis_dataset.parquet'
df_clean.to_parquet(analysis_path, index=False)
stage_message(f'Saved {analysis_path}', started)
clean_summary

In [ ]:
dist_attack = class_distribution(df_clean, 'target_attack')
dist_binary = class_distribution(df_clean, 'target_binary')
dist_attack.to_csv(PROJECT_ROOT / 'outputs/class_distribution_attack.csv', index=False)
dist_binary.to_csv(PROJECT_ROOT / 'outputs/class_distribution_binary.csv', index=False)
dist_attack.head(20)

In [ ]:
# Create DuckDB database for fast SQL queries over the cleaned Parquet.
try:
    import duckdb
    db_path = PROJECT_ROOT / 'data/database/sentinelflow.duckdb'
    con = duckdb.connect(str(db_path))
    con.execute('CREATE OR REPLACE VIEW netflow_clean AS SELECT * FROM read_parquet(?)', [str(analysis_path)])
    profile = con.execute('SELECT COUNT(*) AS rows FROM netflow_clean').fetchdf()
    print('DuckDB created:', db_path)
    display(profile)
    con.close()
except Exception as exc:
    print('DuckDB creation skipped or failed:', exc)

In [ ]:
summary = {
    'raw_dataset': str(raw_path),
    'raw_shape': list(df_raw.shape),
    'clean_shape': list(df_clean.shape),
    'analysis_parquet': str(analysis_path),
    'raw_parquet': str(raw_parquet_path),
    'class_distribution_attack_top10': dist_attack.head(10).to_dict(orient='records'),
    'cleaning_summary': clean_summary,
}
profile_path = PROJECT_ROOT / 'outputs/database_profile.json'
profile_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
html = f"""
<html><head><title>SentinelFlow Database Preparation Summary</title></head><body>
<h1>SentinelFlow Database Preparation Summary</h1>
<p><b>Raw dataset:</b> {raw_path}</p>
<p><b>Raw shape:</b> {df_raw.shape}</p>
<p><b>Clean shape:</b> {df_clean.shape}</p>
<h2>Top Classes</h2>{dist_attack.head(20).to_html(index=False)}
</body></html>
"""
report_path = PROJECT_ROOT / 'reports/database_preparation_summary.html'
report_path.write_text(html, encoding='utf-8')
print('Saved:', profile_path)
print('Saved:', report_path)